In [1]:
import polars as pl
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score

In [2]:
train_df = pl.read_parquet("C:/CreditScoringMLE/ml_service/app/data/train_features.parquet")
test_df = pl.read_parquet("C:/CreditScoringMLE/ml_service/app/data/test_features.parquet")

In [3]:
feature_cols = [c for c in train_df.columns if c not in ("sk_id_curr", "target")]
X = train_df.select(feature_cols).to_pandas().copy()
y = train_df["target"].to_pandas().copy()

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print(X_train.shape, X_val.shape)

(246008, 197) (61503, 197)


In [4]:
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"scale_pos_weight: {scale_pos_weight:.2f}")

model = xgb.XGBClassifier(
    n_estimators=5000,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    eval_metric="auc",
    early_stopping_rounds=100,
    tree_method="hist",
    random_state=42,
)

model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=100,
)

scale_pos_weight: 11.39
[0]	validation_0-auc:0.71771
[100]	validation_0-auc:0.76576
[200]	validation_0-auc:0.77170
[300]	validation_0-auc:0.77414
[400]	validation_0-auc:0.77496
[500]	validation_0-auc:0.77526
[600]	validation_0-auc:0.77509
[606]	validation_0-auc:0.77494


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=100,
              enable_categorical=False, eval_metric='auc', feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.05, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=5000,
              n_jobs=None, num_parallel_tree=None, ...)

In [5]:
val_proba = model.predict_proba(X_val)[:, 1]

roc_auc = roc_auc_score(y_val, val_proba)
pr_auc = average_precision_score(y_val, val_proba)

print(f"ROC-AUC: {roc_auc:.4f}")
print(f"PR-AUC: {pr_auc:.4f}")
print(f"Лучшая итерация: {model.best_iteration}")

ROC-AUC: 0.7755
PR-AUC: 0.2695
Лучшая итерация: 506


In [6]:
importance_df = pl.DataFrame({
    "feature": X_train.columns.tolist(),
    "importance": model.feature_importances_,
}).sort("importance", descending=True)

print(importance_df.head(20))

shape: (20, 2)
┌─────────────────────────────────┬────────────┐
│ feature                         ┆ importance │
│ ---                             ┆ ---        │
│ str                             ┆ f32        │
╞═════════════════════════════════╪════════════╡
│ ext_source_3                    ┆ 0.029188   │
│ ext_source_2                    ┆ 0.02788    │
│ name_education_type_Higher edu… ┆ 0.022253   │
│ code_gender_F                   ┆ 0.015183   │
│ code_gender_M                   ┆ 0.01339    │
│ …                               ┆ …          │
│ name_family_status_Married      ┆ 0.007951   │
│ ext_source_1_was_missing        ┆ 0.007893   │
│ organization_type_te            ┆ 0.00772    │
│ def_60_cnt_social_circle        ┆ 0.007568   │
│ flag_emp_phone                  ┆ 0.007528   │
└─────────────────────────────────┴────────────┘


In [7]:
inst_df = pl.read_csv("C:/CreditScoringMLE/ml_service/app/data/installments_payments.csv")
inst_df = inst_df.select(pl.all().name.to_lowercase())

inst_df = inst_df.with_columns([
    (pl.col("days_entry_payment") - pl.col("days_instalment")).alias("payment_delay_days"),
    (pl.col("amt_instalment") - pl.col("amt_payment")).alias("payment_shortfall"),
])

inst_agg = inst_df.group_by("sk_id_curr").agg([
    pl.col("payment_delay_days").mean().alias("inst_avg_delay_days"),
    pl.col("payment_delay_days").max().alias("inst_max_delay_days"),
    (pl.col("payment_delay_days") > 0).mean().alias("inst_late_payment_rate"),
    pl.col("payment_shortfall").mean().alias("inst_avg_shortfall"),
    pl.len().alias("inst_payments_cnt"),
])

train_df = train_df.join(inst_agg, left_on="sk_id_curr", right_on="sk_id_curr", how="left")
test_df = test_df.join(inst_agg, left_on="sk_id_curr", right_on="sk_id_curr", how="left")

In [8]:
pos_df = pl.read_csv("C:/CreditScoringMLE/ml_service/app/data/POS_CASH_balance.csv")
pos_df = pos_df.select(pl.all().name.to_lowercase())

pos_agg = pos_df.group_by("sk_id_curr").agg([
    pl.col("sk_dpd").mean().alias("pos_avg_dpd"),
    pl.col("sk_dpd").max().alias("pos_max_dpd"),
    pl.col("cnt_instalment_future").mean().alias("pos_cnt_instalment_future_mean"),
])

train_df = train_df.join(pos_agg, left_on="sk_id_curr", right_on="sk_id_curr", how="left")
test_df = test_df.join(pos_agg, left_on="sk_id_curr", right_on="sk_id_curr", how="left")

In [9]:
zero_fill = ["inst_payments_cnt", "inst_late_payment_rate"]
train_df = train_df.with_columns([pl.col(c).fill_null(0) for c in zero_fill])
test_df = test_df.with_columns([pl.col(c).fill_null(0) for c in zero_fill])

median_fill = ["inst_avg_delay_days", "inst_max_delay_days", "inst_avg_shortfall",
               "pos_avg_dpd", "pos_max_dpd", "pos_cnt_instalment_future_mean"]
for c in median_fill:
    med = train_df[c].median()
    train_df = train_df.with_columns(pl.col(c).fill_null(med))
    test_df = test_df.with_columns(pl.col(c).fill_null(med))

In [10]:
train_df.write_parquet("C:/CreditScoringMLE/ml_service/app/data/train_features.parquet")
test_df.write_parquet("C:/CreditScoringMLE/ml_service/app/data/test_features.parquet")

In [11]:
new_cols = ["inst_avg_delay_days", "inst_max_delay_days", "inst_late_payment_rate",
            "inst_avg_shortfall", "inst_payments_cnt", "pos_avg_dpd", "pos_max_dpd",
            "pos_cnt_instalment_future_mean"]

print(train_df.select(new_cols).null_count())
print(train_df.select(new_cols).describe())

shape: (1, 8)
┌────────────┬────────────┬────────────┬───────────┬───────────┬───────────┬───────────┬───────────┐
│ inst_avg_d ┆ inst_max_d ┆ inst_late_ ┆ inst_avg_ ┆ inst_paym ┆ pos_avg_d ┆ pos_max_d ┆ pos_cnt_i │
│ elay_days  ┆ elay_days  ┆ payment_ra ┆ shortfall ┆ ents_cnt  ┆ pd        ┆ pd        ┆ nstalment │
│ ---        ┆ ---        ┆ te         ┆ ---       ┆ ---       ┆ ---       ┆ ---       ┆ _future_m │
│ u32        ┆ u32        ┆ ---        ┆ u32       ┆ u32       ┆ u32       ┆ u32       ┆ ean       │
│            ┆            ┆ u32        ┆           ┆           ┆           ┆           ┆ ---       │
│            ┆            ┆            ┆           ┆           ┆           ┆           ┆ u32       │
╞════════════╪════════════╪════════════╪═══════════╪═══════════╪═══════════╪═══════════╪═══════════╡
│ 0          ┆ 0          ┆ 0          ┆ 0         ┆ 0         ┆ 0         ┆ 0         ┆ 0         │
└────────────┴────────────┴────────────┴───────────┴───────────┴───────────┴─

In [12]:
train_df = pl.read_parquet("C:/CreditScoringMLE/ml_service/app/data/train_features.parquet")

feature_cols = [c for c in train_df.columns if c not in ("sk_id_curr", "target")]
X = train_df.select(feature_cols).to_pandas().copy()
y = train_df["target"].to_pandas().copy()

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

In [16]:
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"scale_pos_weight: {scale_pos_weight:.2f}")

model = xgb.XGBClassifier(
    n_estimators=5000,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    eval_metric="auc",
    early_stopping_rounds=100,
    tree_method="hist",
    random_state=42,
)

model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=100,
)

scale_pos_weight: 11.39
[0]	validation_0-auc:0.68915
[100]	validation_0-auc:0.77253
[200]	validation_0-auc:0.77741
[300]	validation_0-auc:0.78019
[400]	validation_0-auc:0.78120


KeyboardInterrupt: 

In [14]:
val_proba = model.predict_proba(X_val)[:, 1]

roc_auc = roc_auc_score(y_val, val_proba)
pr_auc = average_precision_score(y_val, val_proba)

print(f"ROC-AUC: {roc_auc:.4f}")
print(f"PR-AUC: {pr_auc:.4f}")
print(f"Лучшая итерация: {model.best_iteration}")

NotFittedError: need to call fit or load_model beforehand

In [ ]:
importance_df = pl.DataFrame({
    "feature": X_train.columns.tolist(),
    "importance": model.feature_importances_,
}).sort("importance", descending=True)

print(importance_df.head(20))
print(importance_df.filter(pl.col("feature").str.starts_with("inst_") | pl.col("feature").str.starts_with("pos_")))

shape: (20, 2)
┌─────────────────────────────────┬────────────┐
│ feature                         ┆ importance │
│ ---                             ┆ ---        │
│ str                             ┆ f32        │
╞═════════════════════════════════╪════════════╡
│ ext_source_3                    ┆ 0.029871   │
│ ext_source_2                    ┆ 0.028932   │
│ name_education_type_Higher edu… ┆ 0.020391   │
│ code_gender_M                   ┆ 0.017237   │
│ name_contract_type_Revolving l… ┆ 0.01489    │
│ …                               ┆ …          │
│ region_rating_client_w_city     ┆ 0.008191   │
│ name_contract_type_Cash loans   ┆ 0.007915   │
│ organization_type_te            ┆ 0.007885   │
│ prev_app_approved_cnt           ┆ 0.007839   │
│ def_60_cnt_social_circle        ┆ 0.007642   │
└─────────────────────────────────┴────────────┘
shape: (8, 2)
┌────────────────────────────────┬────────────┐
│ feature                        ┆ importance │
│ ---                            ┆ ---    

In [ ]:
# cc_df = pl.read_csv("C:/CreditScoringMLE/ml_service/app/data/credit_card_balance.csv")
# cc_df = cc_df.select(pl.all().name.to_lowercase())

# cc_df = cc_df.with_columns(
#     (pl.col("amt_balance") / pl.col("amt_credit_limit_actual")).alias("cc_utilization")
# )

# cc_agg = cc_df.group_by("sk_id_curr").agg([
#     pl.col("amt_balance").mean().alias("cc_avg_balance"),
#     pl.col("cc_utilization").mean().alias("cc_avg_utilization"),
#     pl.col("cc_utilization").max().alias("cc_max_utilization"),
#     pl.col("sk_dpd").mean().alias("cc_avg_dpd"),
#     pl.col("sk_dpd").max().alias("cc_max_dpd"),
#     pl.len().alias("cc_records_cnt"),
# ])

# train_df = train_df.join(cc_agg, left_on="sk_id_curr", right_on="sk_id_curr", how="left")
# test_df = test_df.join(cc_agg, left_on="sk_id_curr", right_on="sk_id_curr", how="left")

In [ ]:
n_inf = train_df.filter(pl.col("cc_avg_utilization").is_infinite()).height
print(f"inf в cc_avg_utilization: {n_inf}")

train_df = train_df.with_columns([
    pl.when(pl.col(c).is_infinite()).then(None).otherwise(pl.col(c)).alias(c)
    for c in ["cc_avg_utilization", "cc_max_utilization"]
])
test_df = test_df.with_columns([
    pl.when(pl.col(c).is_infinite()).then(None).otherwise(pl.col(c)).alias(c)
    for c in ["cc_avg_utilization", "cc_max_utilization"]
])

inf в cc_avg_utilization: 197


In [ ]:
zero_fill_cc = ["cc_records_cnt"]
train_df = train_df.with_columns([pl.col(c).fill_null(0) for c in zero_fill_cc])
test_df = test_df.with_columns([pl.col(c).fill_null(0) for c in zero_fill_cc])

median_fill_cc = ["cc_avg_balance", "cc_avg_utilization", "cc_max_utilization", "cc_avg_dpd", "cc_max_dpd"]
for c in median_fill_cc:
    med = train_df[c].median()
    train_df = train_df.with_columns(pl.col(c).fill_null(med))
    test_df = test_df.with_columns(pl.col(c).fill_null(med))

In [ ]:
train_df = pl.read_parquet("C:/CreditScoringMLE/ml_service/app/data/train_features.parquet")

feature_cols = [c for c in train_df.columns if c not in ("sk_id_curr", "target")]
X = train_df.select(feature_cols).to_pandas().copy()
y = train_df["target"].to_pandas().copy()

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

model = xgb.XGBClassifier(
    n_estimators=5000, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight, eval_metric="auc",
    early_stopping_rounds=100, tree_method="hist", random_state=42,
)
model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=100)

val_proba = model.predict_proba(X_val)[:, 1]
print(f"ROC-AUC: {roc_auc_score(y_val, val_proba):.4f}")
print(f"PR-AUC: {average_precision_score(y_val, val_proba):.4f}")

[0]	validation_0-auc:0.68915
[100]	validation_0-auc:0.77253
[200]	validation_0-auc:0.77741
[300]	validation_0-auc:0.78019
[400]	validation_0-auc:0.78120
[500]	validation_0-auc:0.78139
[534]	validation_0-auc:0.78151
ROC-AUC: 0.7817
PR-AUC: 0.2764


In [ ]:
# import optuna

# def objective(trial):
#     params = {
#         "n_estimators": 3000,
#         "max_depth": trial.suggest_int("max_depth", 3, 10),
#         "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
#         "subsample": trial.suggest_float("subsample", 0.6, 1.0),
#         "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
#         "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
#         "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10, log=True),
#         "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10, log=True),
#         "scale_pos_weight": scale_pos_weight,
#         "eval_metric": "auc",
#         "tree_method": "hist",
#         "random_state": 42,
#     }
#     m = xgb.XGBClassifier(**params, early_stopping_rounds=50)
#     m.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
#     proba = m.predict_proba(X_val)[:, 1]
#     return roc_auc_score(y_val, proba)

# study = optuna.create_study(direction="maximize")
# study.optimize(objective, n_trials=25, show_progress_bar=True)

# print("Best ROC-AUC:", study.best_value)
# print("Best params:", study.best_params)

c:\CreditScoringMLE\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2026-08-11 12:10:17,285] A new study created in memory with name: no-name-e18dc4e2-43cb-429c-85ad-ac55aa7d6684
Best trial: 0. Best value: 0.782934:   4%|▍         | 1/25 [01:57<47:05, 117.75s/it]

[I 2026-08-11 12:12:15,028] Trial 0 finished with value: 0.7829339121061695 and parameters: {'max_depth': 6, 'learning_rate': 0.010379422728120884, 'subsample': 0.9601201050742718, 'colsample_bytree': 0.9687800271599037, 'min_child_weight': 8, 'reg_alpha': 2.107031476417174, 'reg_lambda': 0.03743767068151615}. Best is trial 0 with value: 0.7829339121061695.


Best trial: 0. Best value: 0.782934:   8%|▊         | 2/25 [02:18<23:18, 60.80s/it] 

[I 2026-08-11 12:12:35,993] Trial 1 finished with value: 0.7820409693707593 and parameters: {'max_depth': 5, 'learning_rate': 0.0680964787559464, 'subsample': 0.9710798390267851, 'colsample_bytree': 0.6168807487292852, 'min_child_weight': 6, 'reg_alpha': 0.17586388129837682, 'reg_lambda': 0.014401549009555705}. Best is trial 0 with value: 0.7829339121061695.


Best trial: 0. Best value: 0.782934:  12%|█▏        | 3/25 [03:09<20:38, 56.29s/it]

...

[I 2026-08-11 12:36:27,902] Trial 22 finished with value: 0.7844178341745361 and parameters: {'max_depth': 4, 'learning_rate': 0.018518010212483425, 'subsample': 0.9143478993515927, 'colsample_bytree': 0.723192338036526, 'min_child_weight': 2, 'reg_alpha': 0.0021014274049007083, 'reg_lambda': 1.5335170536891005}. Best is trial 14 with value: 0.7845963272498205.


Best trial: 23. Best value: 0.785178:  96%|█████████▌| 24/25 [29:10<01:51, 111.91s/it]

[I 2026-08-11 12:39:27,754] Trial 23 finished with value: 0.7851783436334221 and parameters: {'max_depth': 4, 'learning_rate': 0.013871543829912467, 'subsample': 0.8300940975642206, 'colsample_bytree': 0.73528659446804, 'min_child_weight': 3, 'reg_alpha': 0.0027393808647190316, 'reg_lambda': 0.5754807300169648}. Best is trial 23 with value: 0.7851783436334221.


Best trial: 23. Best value: 0.785178: 100%|██████████| 25/25 [31:28<00:00, 75.54s/it] 

[I 2026-08-11 12:41:45,871] Trial 24 finished with value: 0.7840705875722723 and parameters: {'max_depth': 5, 'learning_rate': 0.014826239416355495, 'subsample': 0.8385125781245217, 'colsample_bytree': 0.7949663906745241, 'min_child_weight': 3, 'reg_alpha': 0.007532727877075818, 'reg_lambda': 0.39493293770248844}. Best is trial 23 with value: 0.7851783436334221.
Best ROC-AUC: 0.7851783436334221
Best params: {'max_depth': 4, 'learning_rate': 0.013871543829912467, 'subsample': 0.8300940975642206, 'colsample_bytree': 0.73528659446804, 'min_child_weight': 3, 'reg_alpha': 0.0027393808647190316, 'reg_lambda': 0.5754807300169648}


In [ ]:
best_params = study.best_params
best_params.update({
    "n_estimators": 5000,
    "scale_pos_weight": scale_pos_weight,
    "eval_metric": "auc",
    "tree_method": "hist",
    "random_state": 42,
})

final_model = xgb.XGBClassifier(**best_params, early_stopping_rounds=100)
final_model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=100)

val_proba = final_model.predict_proba(X_val)[:, 1]
print(f"Final ROC-AUC: {roc_auc_score(y_val, val_proba):.4f}")
print(f"Final PR-AUC: {average_precision_score(y_val, val_proba):.4f}")

import joblib
joblib.dump(final_model, "C:/CreditScoringMLE/ml_service/app/models/xgboost_model.joblib")

[0]	validation_0-auc:0.67550
[100]	validation_0-auc:0.74122
[200]	validation_0-auc:0.75505
[300]	validation_0-auc:0.76366
[400]	validation_0-auc:0.76863
[500]	validation_0-auc:0.77172
[600]	validation_0-auc:0.77393
[700]	validation_0-auc:0.77562
[800]	validation_0-auc:0.77694
[900]	validation_0-auc:0.77804
[1000]	validation_0-auc:0.77878
[1100]	validation_0-auc:0.77944
[1200]	validation_0-auc:0.78001
[1300]	validation_0-auc:0.78065
[1400]	validation_0-auc:0.78126
[1500]	validation_0-auc:0.78167
[1600]	validation_0-auc:0.78217
[1700]	validation_0-auc:0.78253
[1800]	validation_0-auc:0.78284
[1900]	validation_0-auc:0.78324
[2000]	validation_0-auc:0.78359
[2100]	validation_0-auc:0.78370
[2200]	validation_0-auc:0.78390
[2300]	validation_0-auc:0.78409
[2400]	validation_0-auc:0.78430
[2500]	validation_0-auc:0.78454
[2600]	validation_0-auc:0.78471
[2700]	validation_0-auc:0.78482
[2800]	validation_0-auc:0.78497
[2900]	validation_0-auc:0.78507
[3000]	validation_0-auc:0.78518
[3100]	validation_0-

['C:/CreditScoringMLE/ml_service/app/models/xgboost_model.joblib']

In [ ]:
import json

booster = final_model.get_booster()
feature_order = booster.feature_names  # список ИЗ САМОЙ модели — источник истины

defaults = {}
for c in feature_order:
    if c in X_train.columns:
        defaults[c] = float(X_train[c].median())
    else:
        defaults[c] = 0.0  # на случай, если такой колонки в X_train вообще нет

with open("C:/CreditScoringMLE/ml_service/app/models/feature_defaults.json", "w") as f:
    json.dump({"feature_order": feature_order, "defaults": defaults}, f, indent=2)

print(f"Сохранено {len(feature_order)} признаков")

NameError: name 'final_model' is not defined